# Embeddings for the DDIS Movie Graph

In [39]:
CONFIG = {
    "Hosting": {
        "URL": "https://speakeasy.ifi.uzh.ch",
        "Username": "CyanPeekingMouse",
        "Password": "Qe5Hf3zJ"
    },
    "Data": {
        "Download_URL": "https://files.ifi.uzh.ch/ddis/teaching/2025/ATAI/dataset/",
        "Directory": "dataset/",
        "Cache": "cache/",
    }
}

## Setup

In [40]:
# imports
import csv
import numpy as np
import os
import rdflib
from rdflib import Graph
import pandas as pd
from sklearn.metrics import pairwise_distances
import json

In [41]:
# define some prefixes
WD = rdflib.Namespace('http://www.wikidata.org/entity/')
WDT = rdflib.Namespace('http://www.wikidata.org/prop/direct/')
DDIS = rdflib.Namespace('http://ddis.ch/atai/')
RDFS = rdflib.namespace.RDFS
SCHEMA = rdflib.Namespace('http://schema.org/')

## Load the data

In [42]:
graph = Graph()
data_dir = CONFIG["Data"]["Directory"]

graph_path = os.path.join(data_dir, "graph.nt")
if os.path.exists(graph_path):
    print(f"Loading graph at {graph_path}...")
    graph.parse(graph_path, format="nt")
    print(f"Loaded {graph_path} → {len(graph)} triples")
else:
    print(f"Graph does not exist at {graph_path}")

Loading graph at dataset/graph.nt...
Loaded dataset/graph.nt → 2413825 triples


In [43]:
# load the embeddings
entity_emb = np.load(os.path.join(data_dir, 'embeddings', 'entity_embeds.npy'))
relation_emb = np.load(os.path.join(data_dir, 'embeddings', 'relation_embeds.npy'))

In [44]:
cache_dir = CONFIG["Data"]["Cache"]

# id: index
# ent: entity identifier
# rel: relation identifier
# lbl: label

# load the entities dictionaries
with open(os.path.join(cache_dir, 'entities', 'identifier_to_index.json'), "r") as f:
    # identifier -> index mapping
    ent2id = json.load(f)
    # index -> identifier mapping
    id2ent = {v: k for k, v in ent2id.items()}

with open(os.path.join(cache_dir, 'entities', 'identifier_to_label.json'), "r") as f:
    # identifier -> label mapping
    ent2lbl = json.load(f)
    # label -> identifier mapping
    lbl2ent = {lbl: ent for ent, lbl in ent2lbl.items()}

print(ent2id["http://www.wikidata.org/entity/Q193835"])
print(id2ent[54250])
print(ent2lbl["http://www.wikidata.org/entity/Q193835"])
print(lbl2ent["Good Will Hunting"])
print(ent2lbl[id2ent[54250]])

# load the relations dictionaries
with open(os.path.join(cache_dir, 'relations', 'identifier_to_index.json'), "r") as f:
    # identifier -> index mapping
    rel2id = json.load(f)
    # index -> identifier mapping
    id2rel = {v: k for k, v in ent2id.items()}

with open(os.path.join(cache_dir, 'relations', 'identifier_to_label.json'), "r") as f:
    # identifier -> label mapping
    rel2lbl = json.load(f)
    # label -> identifier mapping
    lbl2rel = {lbl: ent for ent, lbl in ent2lbl.items()}

54250
http://www.wikidata.org/entity/Q193835
Good Will Hunting
http://www.wikidata.org/entity/Q193835
Good Will Hunting


## Inspect the data

In [45]:
# number of triples in the graph
len(graph)

2413825

In [46]:
# number of entities in the graph
triples = {(s, p, o) for s,p,o in graph.triples((None, None, None)) if isinstance(o, rdflib.term.URIRef)}
len({s for s,p,o in triples} | {o for s,p,o in triples})

175660

In [47]:
# entity embedding size
entity_emb.shape

(175660, 256)

In [48]:
# relation embedding size
relation_emb.shape

(491, 256)

## Finding errors

In [49]:
# let's see what our graph thinks the occupation of Jean Van Hamme is
professions = set(graph.query('''
    prefix wdt: <http://www.wikidata.org/prop/direct/>
    prefix wd: <http://www.wikidata.org/entity/>
    
    SELECT ?obj ?lbl WHERE {
        ?ent rdfs:label "Good Will Hunting"@en .
        ?ent wdt:P57 ?obj .
        ?obj rdfs:label ?lbl .
    }
    '''))
{ent[len(WD):]: str(lbl) for ent, lbl in professions}

{}

In [50]:
# "Good Will Hunting" entity
head = entity_emb[ent2id["http://www.wikidata.org/entity/Q193835"]]
# "director" relation
pred = relation_emb[rel2id["http://www.wikidata.org/prop/direct/P57"]]
# add vectors according to TransE scoring function.
lhs = head + pred
# compute distance to *any* entity
dist = pairwise_distances(lhs.reshape(1, -1), entity_emb).reshape(-1)
# find most plausible entities
most_likely = dist.argsort()
# compute ranks of entities
ranks = dist.argsort().argsort()

In [51]:
# show scores for (Jean Van Hamme, occupation, butcher)
pd.DataFrame([(str(lbl), dist[ent2id[ent]], ranks[ent2id[ent]]) for ent, lbl in professions],
        columns=('Occupation', 'Score', 'Rank'))

,Occupation,Score,Rank


In [52]:
# what would be more plausible occupations?
pd.DataFrame([
    (id2ent[idx][len(WD):], ent2lbl[id2ent[idx]], dist[idx], rank+1)
    for rank, idx in enumerate(most_likely[:10])],
    columns=('Entity', 'Label', 'Score', 'Rank'))

,Entity,Label,Score,Rank
0,Q187364,Robert Zemeckis,5.775041e-07,1
1,Q83338,Robin Williams,5.941809e-07,2
2,Q2469007,Tuesday Knight,5.973489e-07,3
3,Q2263,Tom Hanks,5.992046e-07,4
4,Q165219,Robert Downey Jr.,6.003152e-07,5
5,Q244234,Juhn Turturro,6.017929e-07,6
6,Q508904,"Sergey Bodrov, Jr.",6.120370e-07,7
7,Q55424,Peter Weir,6.129433e-07,8
8,Q2959515,Charles K. French,6.140291e-07,9
9,Q313388,Jonah Hill,6.154739e-07,10


## Entity Similarity

In [53]:
# which entities are similar to "Harry Potter and the Goblet of Fire"
ent = ent2id["http://www.wikidata.org/entity/Q88532913"]
print(ent)
# we compare the embedding of the query entity to all other entity embeddings
dist = pairwise_distances(entity_emb[ent].reshape(1, -1), entity_emb).reshape(-1)
# order by plausibility
most_likely = dist.argsort()

pd.DataFrame([
    (
        id2ent[idx][len(WD):], # qid
        ent2lbl.get(id2ent[idx], None),  # label
        dist[idx],             # score
        rank+1,                # rank
    )
    for rank, idx in enumerate(most_likely[:15])],
    columns=('Entity', 'Label', 'Score', 'Rank'))

167028


,Entity,Label,Score,Rank
0,Q88532913,The Beatles: Get Back,0.000000,1
1,Q128730,Thirteen Days,0.000002,2
2,Q1123463,The Man in the Iron Mask,0.000002,3
3,Q18199330,Fantastic Beasts and Where to Find Them,0.000002,4
4,Q109301770,Ennio,0.000002,5
5,Q1196918,The Son,0.000002,6
6,Q162518,How the West Was Won,0.000002,7
7,Q707986,The Diary of Anne Frank,0.000002,8
8,Q19702971,Land of Mine,0.000002,9
9,Q309246,The Motorcycle Diaries,0.000002,10


## Recovering categories

In [54]:
# hmm, our graph contains no parent class of bridge (Q12280)...
set(graph.objects(WD.Q12280, WDT.P279))

set()

In [55]:
# maybe an indirect subclass?
set(graph.objects(WD.Q12280, DDIS.indirectSubclassOf))

set()

In [59]:
# Let's analyze why we're getting unexpected results
print("=== Actual data from graph ===")
# Query to get the actual director of Good Will Hunting
actual_director = list(graph.query('''
    prefix wdt: <http://www.wikidata.org/prop/direct/>
    prefix wd: <http://www.wikidata.org/entity/>
    
    SELECT ?director ?name WHERE {
        wd:Q193835 wdt:P57 ?director .
        ?director rdfs:label ?name .
    }
    '''))
print(f"Actual director from graph: {actual_director[0][1] if actual_director else 'Not found'}\n")

# Now let's look at embedding predictions
print("=== Embedding predictions ===")
# "Good Will Hunting" entity
head = entity_emb[ent2id["http://www.wikidata.org/entity/Q193835"]]
# "director" relation
pred = relation_emb[rel2id["http://www.wikidata.org/prop/direct/P57"]]
# add vectors according to the TransE scoring function
lhs = head + pred
# compute distance to *any* entity
dist = pairwise_distances(lhs.reshape(1, -1), entity_emb).reshape(-1)
# find most plausible tails
most_likely = dist.argsort()

# Show top 5 predictions with scores
results = []
for rank, idx in enumerate(most_likely[:5]):
    entity_uri = id2ent[idx]
    entity_id = entity_uri[len(WD):]
    label = ent2lbl.get(entity_uri, f"Unknown ({entity_id})")
    score = dist[idx]
    results.append((entity_id, label, score, rank + 1))

print("\nTop 5 predictions from embeddings:")
df = pd.DataFrame(results, columns=('Entity', 'Label', 'Score', 'Rank'))
print(df)

# Let's also check Gus Van Sant's position in the predictions
if "http://www.wikidata.org/entity/Q60786" in ent2id:  # Gus Van Sant's Q ID
    gus_idx = ent2id["http://www.wikidata.org/entity/Q60786"]
    gus_rank = (dist.argsort() == gus_idx).nonzero()[0][0] + 1
    print(f"\nGus Van Sant's rank in predictions: {gus_rank}")
    print(f"Score for Gus Van Sant: {dist[gus_idx]}")

=== Actual data from graph ===
Actual director from graph: Gus Van Sant

=== Embedding predictions ===

Top 5 predictions from embeddings:
     Entity              Label         Score  Rank
0   Q187364    Robert Zemeckis  5.775041e-07     1
1    Q83338     Robin Williams  5.941809e-07     2
2  Q2469007     Tuesday Knight  5.973489e-07     3
3     Q2263          Tom Hanks  5.992046e-07     4
4   Q165219  Robert Downey Jr.  6.003152e-07     5


In [60]:
# Let's analyze the movie-actor relationships to understand the predictions
print("=== Movie-Actor Relationships ===")
actors = list(graph.query('''
    prefix wdt: <http://www.wikidata.org/prop/direct/>
    prefix wd: <http://www.wikidata.org/entity/>
    
    SELECT ?actor ?name WHERE {
        wd:Q193835 wdt:P161 ?actor .  # P161 is "cast member"
        ?actor rdfs:label ?name .
    }
    '''))
print("Actors in Good Will Hunting:")
for actor in actors:
    print(f"- {actor[1]}")

print("\n=== Directors who worked with Robin Williams ===")
robin_movies = list(graph.query('''
    prefix wdt: <http://www.wikidata.org/prop/direct/>
    prefix wd: <http://www.wikidata.org/entity/>
    
    SELECT DISTINCT ?director ?name WHERE {
        ?movie wdt:P161 wd:Q83338 .  # Movies with Robin Williams
        ?movie wdt:P57 ?director .    # Their directors
        ?director rdfs:label ?name .
    }
    '''))
print("Directors who worked with Robin Williams:")
for director in robin_movies:
    print(f"- {director[1]}")

=== Movie-Actor Relationships ===
Actors in Good Will Hunting:
- Matt Damon
- Cole Hauser
- Stullan Skirsgård
- Minnie Driver
- Casey Affleck
- Scott William Winters
- Robin Williams
- Harmony Korine
- George Plimpton
- Vik Sahay
- Alison Folland
- Ben Affleck

=== Directors who worked with Robin Williams ===
Directors who worked with Robin Williams:
- George Roy Hill
- Tom Shadyac
- Harold Ramis
- Shawn Levy
- Chris Columbus
- Barry Levinson
- Robert Altman
- Terry Gilliam
- Ivan Reitman
- Christopher Nolan
- Mike Nichols
- Fielder Cook
- Peter Weir
- Vincent Ward
- Gus Van Sant
- Steven Spielberg
- Paul Mazursky
- Joe Johnston
- Kirsten Sheridan
- Woody Allen
- Richard Fleischer
- Kenneth Branagh
- Les Mayfield
- Roger Donaldson
- Penny Marshall
- Alain Cuniot


In [61]:
# Analyze the current embedding space
print("=== Embedding Statistics ===")
print(f"Entity embedding dimensions: {entity_emb.shape}")
print(f"Relation embedding dimensions: {relation_emb.shape}")

# Calculate some basic statistics about the embeddings
print("\nEntity Embedding Statistics:")
print(f"Mean norm: {np.mean(np.linalg.norm(entity_emb, axis=1)):.4f}")
print(f"Std norm: {np.std(np.linalg.norm(entity_emb, axis=1)):.4f}")
print(f"Min norm: {np.min(np.linalg.norm(entity_emb, axis=1)):.4f}")
print(f"Max norm: {np.max(np.linalg.norm(entity_emb, axis=1)):.4f}")

# Get statistics about the relations in the graph
print("\n=== Relation Statistics ===")
relation_counts = {}
for s, p, o in graph:
    if isinstance(p, rdflib.term.URIRef) and isinstance(o, rdflib.term.URIRef):
        relation_counts[p] = relation_counts.get(p, 0) + 1

print(f"Total number of relations: {len(relation_counts)}")
print("\nTop 5 most common relations:")
for rel, count in sorted(relation_counts.items(), key=lambda x: x[1], reverse=True)[:5]:
    print(f"- {rel}: {count} occurrences")

=== Embedding Statistics ===
Entity embedding dimensions: (175660, 256)
Relation embedding dimensions: (491, 256)

Entity Embedding Statistics:
Mean norm: 1.0000
Std norm: 0.0000
Min norm: 1.0000
Max norm: 1.0000

=== Relation Statistics ===
Total number of relations: 490

Top 5 most common relations:
- http://www.wikidata.org/prop/direct/P106: 251851 occurrences
- http://www.wikidata.org/prop/direct/P161: 167525 occurrences
- http://www.wikidata.org/prop/direct/P31: 148686 occurrences
- http://www.wikidata.org/prop/direct/P21: 92707 occurrences
- http://www.wikidata.org/prop/direct/P27: 86511 occurrences
Total number of relations: 490

Top 5 most common relations:
- http://www.wikidata.org/prop/direct/P106: 251851 occurrences
- http://www.wikidata.org/prop/direct/P161: 167525 occurrences
- http://www.wikidata.org/prop/direct/P31: 148686 occurrences
- http://www.wikidata.org/prop/direct/P21: 92707 occurrences
- http://www.wikidata.org/prop/direct/P27: 86511 occurrences


In [62]:
# Let's look at the director relation specifically
director_triples = list(graph.query('''
    prefix wdt: <http://www.wikidata.org/prop/direct/>
    prefix wd: <http://www.wikidata.org/entity/>
    
    SELECT (COUNT(?movie) as ?count) WHERE {
        ?movie wdt:P57 ?director .
    }
    '''))
print(f"Number of director relations (P57): {director_triples[0][0]}")

# Calculate ratio of correct predictions for director relationship
test_movies = list(graph.query('''
    prefix wdt: <http://www.wikidata.org/prop/direct/>
    prefix wd: <http://www.wikidata.org/entity/>
    
    SELECT ?movie ?director WHERE {
        ?movie wdt:P57 ?director .
    } LIMIT 100
    '''))

correct = 0
total = len(test_movies)

for movie, director in test_movies:
    if movie.startswith('http://www.wikidata.org/entity/'):
        head = entity_emb[ent2id[movie]]
        pred = relation_emb[rel2id["http://www.wikidata.org/prop/direct/P57"]]
        lhs = head + pred
        dist = pairwise_distances(lhs.reshape(1, -1), entity_emb).reshape(-1)
        top_pred = id2ent[dist.argsort()[0]]
        if top_pred == director:
            correct += 1

print(f"\nAccuracy of director predictions: {correct/total:.2%}")

Number of director relations (P57): 15187


KeyError: rdflib.term.URIRef('http://www.wikidata.org/entity/Q204416')

## Recommendations for Retraining

1. **Remove Strict Normalization**
- Don't normalize embeddings to unit length during training
- Use L2 regularization instead of strict normalization
- This allows more expressiveness in the embedding space

2. **Relation-Specific Training**
- Use relation-specific margins in the loss function
- Give higher weight to less frequent but important relations like P57 (director)
- Implement negative sampling that accounts for relation frequency

3. **Model Architecture**
- Current dimension (256) is good for the dataset size
- Consider using RotatE or ComplEx instead of TransE
- These models can better handle various relation patterns

In [ ]:
# 4. **Training Process**

# Pseudo-code for improved training:
relation_weights = {
    'P57': 5.0,  # Boost director relationship
    'P161': 0.5, # Reduce cast member weight
    'P106': 0.5  # Reduce occupation weight
}

# Loss function with relation-specific margins and weights
loss = margin_loss + (
    relation_weights[rel_type] * 
    L2_regularization +
    relation_specific_margin
)

# Negative sampling strategy
neg_samples = sample_negatives(
    pos_triple,
    num_samples=5,
    relation_freq=relation_counts
)

# Training Analysis

In [58]:
# ... didn't really help.
# Let's try ddis:indirectSubclassOf next

# set the head entity to bridge
head = entity_emb[ent2id[WD.Q12280]]
# now we try ddis:indirectSubclassOf
pred = relation_emb[rel2id[DDIS.indirectSubclassOf]]
# combine according to the TransE scoring function
lhs = head + pred
# compute distance to *any* entity
dist = pairwise_distances(lhs.reshape(1, -1), entity_emb).reshape(-1)
# find most plausible tails
most_likely = dist.argsort()
# show most likely entities
pd.DataFrame([
    (id2ent[idx][len(WD):], ent2lbl[id2ent[idx]], dist[idx], rank+1)
    for rank, idx in enumerate(most_likely[:10])],
    columns=('Entity', 'Label', 'Score', 'Rank'))

KeyError: rdflib.term.URIRef('http://www.wikidata.org/entity/Q12280')